In [1]:
# -*- coding: utf-8 -*-
"""
《基于隔夜相关性的因子研究》(广发证券, 多因子Alpha系列之五十五, 2025-11-20) —— 聚宽真实数据完整复现版

运行环境：聚宽研究环境(Notebook)整段粘贴运行；本地可设 JQ_USER / JQ_PASS 使用 jqdatasdk。
首次运行强烈建议先把 CFG["QUICK"]=True 跑通（随机800只股票、2021~2023年，几分钟），再关掉QUICK跑全量。

对应研报章节
  三(一) 群组识别        : 直接对"领先/滞后相关性矩阵特征"做KMeans(研报改进方法)，并保留d-LE-SC原方法；
                          按平均流强度 S_{a->b} 确认领先/滞后群组；图7-9(群组数量)、d-LE-SC群组过小的诊断
  三(二) 组合回测        : 沪深300 / 中证1000 / 全市场，三种持仓方案(次日开盘->后日开盘、次日开盘->收盘(日间)、
                          当日收盘->次日收盘(理想))，20%/20%多空，多头组/空头组/多空组/空多组净值，图10-23
  四(一) 回测框架        : 全市场；剔除非上市、ST、涨跌停(一字板)、上市不满1年；MAD去极值+Z-Score；十分档；
                          每周/月最后一个交易日的次日均价(avg)调仓；卖出千分之三
  四(二) 因子与绩效      : 表2/表4/表6/表8/表10，所有因子逐一实现，并与研报数值并列输出
  四(三) 因子相关性      : 表9
  四(四) 组合详情        : 图24-31、表11/12(分年度)、表13(风格因子相关性，聚宽风格因子库代替通联Barra数据)

与研报不可能完全一致的地方(研报未披露或数据源不同，已按最合理方式处理)
  1. 研报未披露回溯窗口、MAD阈值、指数基准：这里 WINDOW=60、MAD=3倍、基准=中证全指，均可在CFG修改；
  2. 研报表4中 signal_5 与 signal_21 结果完全相同，疑似原文实现问题，这里按字面逻辑(5日/21日信号方向)正确实现，故不会相同；
  3. 研报表6中 factor_bottom50_abs 一行与 factor_bottom50 重复，这里按定义(绝对值最小的50只)实现；
  4. 因子方向：IC为负的因子按IC符号翻转后统计多空/多头(与表4 IC为负而多空年化为正一致)，group_1为最优组；
  5. 风格因子：研报用通联数据，这里用聚宽因子库(jqfactor)Barra CNE5风格因子，数值有差异；
  6. 数据区间、成分股、复权口径与研报不同，结果仅能做到"方法与量级可比"。
"""
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib
import matplotlib.pyplot as plt

# ← 新增：旧版numpy没有take_along_axis，补一个仅支持axis=0的等价实现
if not hasattr(np, "take_along_axis"):
    def _take_along_axis(arr, indices, axis=0):
        assert axis == 0
        return arr[indices, np.arange(arr.shape[1])[None, :]]
    np.take_along_axis = _take_along_axis

# ---------------- 聚宽数据接口兼容 ----------------
try:
    from jqdata import *          # 研究环境
except Exception:
    import jqdatasdk as jq        # 本地：pip install jqdatasdk
    jq.auth(os.environ.get("JQ_USER", ""), os.environ.get("JQ_PASS", ""))
    from jqdatasdk import *

try:                              # 聚宽因子库(风格因子)，用于表13
    from jqfactor import get_factor_values as jq_get_factor_values
except Exception:
    try:
        from jqdatasdk import get_factor_values as jq_get_factor_values
    except Exception:
        jq_get_factor_values = None

plt.rcParams["axes.unicode_minus"] = False
for _f in ["SimHei", "Noto Sans CJK SC", "WenQuanYi Zen Hei", "Heiti TC", "Microsoft YaHei"]:
    try:
        matplotlib.font_manager.findfont(_f, fallback_to_default=False)
        plt.rcParams["font.sans-serif"] = [_f]
        break
    except Exception:
        continue

# ==========================================================================
# 配置
# ==========================================================================
CFG = dict(
    DATA_START="2024-01-01", END="2026-10-20",          # ← 改
    DAILY_START="2024-04-01",          # ← 改：需留出60个交易日回溯窗口，不能早于约2024-03下旬
    FACTOR_START="2024-04-01",         # ← 改：同上
    BENCH="000985.XSHG",               # 表11/12"指数收益"基准(研报未披露，取中证全指)
    QUICK=False,                       # True: 随机抽样+短区间，仅用于验证流程
    SAMPLE_N=0,
    ROWS_PER_CALL=30000,               # get_price单次最大行数(股票数*交易日数)，超限会自动二分重试
    RUN_DAILY=True, RUN_FACTOR=True, RUN_STYLE=True,
    DAILY_UNIVERSES=[("中证1000", "000852.XSHG")],
    CLUSTER_METHOD="kmeans",           # "kmeans"(研报改进方法) / "dlesc"(文献原方法)
    CLUSTER_EVERY={"沪深300": 1, "中证1000": 1, "全市场": 5},   # 日度回测中重新聚类的间隔(交易日)；1=每天
    PCA_DIM=10,                        # KMeans前对"领先/滞后特征"做PCA降维的维数
    WINDOW=60,                         # 日度回测相关性矩阵回溯窗口
    FACTOR_WINDOW=60, TOPK=50,
    HALT_MAX_WIN=0.2,                  # 窗口内停牌天数占比上限
    MIN_LIST_DAYS=365,                 # 上市不满1年剔除
    MIN_N_DAILY=60,
    MAD_K=3.0, FEE=0.003, N_GROUP=10,
    OUT_DIR="./outputs",
)
if CFG["QUICK"]:
    CFG.update(DATA_START="2021-01-01", END="2023-12-31", DAILY_START="2021-07-01",
               FACTOR_START="2022-07-01", SAMPLE_N=800, MIN_N_DAILY=40,
               CLUSTER_EVERY={"沪深300": 1, "中证1000": 1, "全市场": 1})
OUT_DIR = CFG["OUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(42)

# 研报数值(用于并列对照)：(RANK_IC%, ICIR, IC胜率%, 多空年化%, 多头年化%)
REPORT = {
    "factor_lead": {"月度": (0.29, 0.03, 49.2, -1.5, 1.4), "周度": (1.30, 0.13, 56.5, 9.5, 3.0)},
    "factor_lag": {"月度": (0.13, 0.01, 49.2, -2.7, 3.5), "周度": (0.54, 0.05, 54.2, 2.9, -2.7)},
    "factor_top50_lead_signal_5": {"月度": (-0.29, -0.03, 49.2, -1.5, 1.4), "周度": (-1.30, -0.13, 56.5, 9.6, 3.0)},
    "factor_top50_lead_signal_21": {"月度": (-0.29, -0.03, 49.2, -1.5, 1.4), "周度": (-1.30, -0.13, 56.5, 9.6, 3.0)},
    "factor_top50_allday_lead_signal_5": {"月度": (-8.80, -0.75, 75.4, 31.9, 16.6), "周度": (-6.09, -0.53, 71.0, 47.1, 16.1)},
    "factor_top50_allday_lead_signal_21": {"月度": (-8.80, -0.75, 75.4, 31.9, 16.6), "周度": (-6.09, -0.53, 71.0, 47.1, 16.1)},
    "corr_lag_21": {"月度": (1.29, 0.11, 55.4, 2.8, 4.5), "周度": (2.35, 0.18, 58.8, 12.6, 5.8)},
    "corr_lag_60": {"月度": (2.68, 0.21, 60.0, 9.4, 8.9), "周度": (2.15, 0.15, 60.3, 14.1, 10.2)},
    "factor_top50": {"月度": (1.09, 0.10, 52.3, 0.8, 4.2), "周度": (1.61, 0.14, 56.1, 9.7, 3.4)},
    "factor_bottom50": {"月度": (0.94, 0.11, 50.8, 3.1, 4.5), "周度": (1.12, 0.11, 51.9, 7.9, 2.9)},
    "factor_net": {"月度": (0.08, 0.01, 56.9, -1.1, 3.7), "周度": (0.61, 0.08, 48.1, 1.9, -0.7)},
    "factor_top50_abs": {"月度": (1.13, 0.10, 47.7, 3.4, 4.9), "周度": (1.64, 0.13, 55.3, 7.8, 4.0)},
    "factor_net_abs": {"月度": (1.12, 0.10, 47.7, 3.1, 4.7), "周度": (1.64, 0.13, 55.3, 7.8, 4.1)},
    "factor_top50_allday": {"月度": (8.11, 0.67, 75.4, 27.6, 18.3), "周度": (6.57, 0.53, 73.3, 50.0, 22.4)},
    "factor_bottom50_allday": {"月度": (4.45, 0.47, 73.8, 14.7, 8.5), "周度": (2.54, 0.28, 66.0, 13.4, 3.2)},
    "factor_net_allday": {"月度": (5.20, 0.42, 67.7, 13.4, 10.3), "周度": (4.73, 0.42, 67.6, 32.1, 15.3)},
    "factor_top50_corr_by_lag": {"月度": (6.65, 0.60, 67.7, 23.2, 14.1), "周度": (5.19, 0.45, 68.3, 33.4, 11.2)},
    "factor_small50_corr_by_lag": {"月度": (5.65, 0.67, 69.2, 19.8, 12.0), "周度": (4.27, 0.43, 66.0, 28.4, 10.1)},
    "corr_combined1": {"月度": (8.13, 0.68, 75.4, 27.6, 18.2), "周度": (6.59, 0.53, 72.9, 49.9, 22.1)},
}
TABLES = {
    "表2_隔夜相关性因子": ["factor_lead", "factor_lag"],
    "表4_方向调整": ["factor_top50_lead_signal_5", "factor_top50_lead_signal_21",
                  "factor_top50_allday_lead_signal_5", "factor_top50_allday_lead_signal_21"],
    "表6_全市场隔夜日间相关性": ["corr_lag_21", "corr_lag_60", "factor_top50", "factor_bottom50", "factor_net",
                       "factor_top50_abs", "factor_bottom50_abs", "factor_net_abs"],
    "表8_常规相关性与结合": ["factor_top50_allday", "factor_bottom50_allday", "factor_net_allday",
                     "factor_top50_corr_by_lag", "factor_small50_corr_by_lag"],
    "表10_组合因子": ["corr_combined1"],
}
FACTOR_NAMES = [n for v in TABLES.values() for n in v]
STYLE_MAP = {"动量": "momentum", "市值": "size", "成长": "growth", "杠杆": "leverage",
             "残差波动": "residual_volatility", "流动性": "liquidity", "盈利": "earnings_yield",
             "贝塔": "beta", "账面市值比": "book_to_price_ratio", "非线性市值": "non_linear_size"}
SCHEMES = {"o2o": "次日开盘买入-后日开盘卖出", "day": "次日开盘买入-当日收盘卖出(日间)", "c2c": "理想：当日收盘买入-次日收盘卖出"}


# ==========================================================================
# 一、数据层：聚宽读取 -> 宽表 -> 隔夜/日间/全天收益 -> 各类过滤掩码
# ==========================================================================
FIELDS = ["open", "close", "avg", "high_limit", "low_limit", "paused"]


def list_all_stocks():
    """多个时点快照取并集，覆盖期间退市股票，缓解幸存者偏差。"""
    snaps = sorted(set([pd.Timestamp(CFG["DATA_START"]), pd.Timestamp(CFG["END"])] +
                       list(pd.date_range(CFG["DATA_START"], CFG["END"], freq="6MS"))))
    frames = [get_all_securities(types=["stock"], date=d.date()) for d in snaps]
    info = pd.concat(frames)
    return info[~info.index.duplicated(keep="last")]


def fetch_prices(codes):
    n_days = len(get_trade_days(start_date=CFG["DATA_START"], end_date=CFG["END"]))
    step = max(1, CFG["ROWS_PER_CALL"] // max(n_days, 1))
    store = {f: [] for f in FIELDS}

    def run(sub):
        try:
            d = get_price(sub, start_date=CFG["DATA_START"], end_date=CFG["END"], frequency="daily",
                          fields=FIELDS, skip_paused=False, fq="pre", panel=False)
        except Exception as e:
            if len(sub) > 1:
                h = len(sub) // 2
                run(sub[:h]); run(sub[h:])
            else:
                print("行情获取失败：", sub, str(e)[:80])
            return
        if d is None or len(d) == 0:
            return
        d["time"] = pd.to_datetime(d["time"])
        d = d.drop_duplicates(subset=["time", "code"])
        for f in FIELDS:
            store[f].append(d.pivot(index="time", columns="code", values=f).astype("float32"))

    for i in range(0, len(codes), step):
        run(codes[i:i + step])
        if (i // step) % 20 == 0:
            print(f"行情进度 {min(i + step, len(codes))}/{len(codes)}")
    return {f: pd.concat(store[f], axis=1).sort_index() for f in FIELDS}


def fetch_st(codes, dates):
    frames = []
    for i in range(0, len(codes), 300):
        try:
            x = get_extras("is_st", codes[i:i + 300], start_date=dates[0], end_date=dates[-1], df=True)
            frames.append(x)
        except Exception as e:
            print("ST数据获取失败(该批视为非ST)：", str(e)[:80])
    if not frames:
        return pd.DataFrame(False, index=dates, columns=codes)
    st = pd.concat(frames, axis=1)
    st.index = pd.to_datetime(st.index)
    return st.reindex(index=dates, columns=codes).fillna(False).astype(bool)


def load_market_data():
    cache = f"{OUT_DIR}/market_cache_{CFG['DATA_START']}_{CFG['END']}_{CFG['SAMPLE_N']}.pkl"
    if os.path.exists(cache):
        print("读取本地缓存", cache)
        return pd.read_pickle(cache)
    info = list_all_stocks()
    codes = list(info.index)
    if CFG["SAMPLE_N"]:
        rng = np.random.RandomState(42)
        codes = list(rng.choice(codes, min(CFG["SAMPLE_N"], len(codes)), replace=False))
    wide = fetch_prices(codes)
    cols = [c for c in codes if c in wide["close"].columns]
    for f in FIELDS:
        wide[f] = wide[f][cols]
    dts = wide["close"].index
    st = fetch_st(cols, dts)
    out = dict(wide=wide, info=info.loc[cols], st=st)
    pd.to_pickle(out, cache)
    return out


t0 = time.time()
RAW = load_market_data()
W_ = RAW["wide"]
dates = W_["close"].index
codes = list(W_["close"].columns)
T, NALL = len(dates), len(codes)
OPEN, CLOSE = W_["open"].values, W_["close"].values
AVG = W_["avg"].values.copy()
PAUSED = ~(W_["paused"].values == 0)   # ← 改：paused为0才算未停牌，NaN(未上市/退市)按停牌处理
HL, LL = W_["high_limit"].values, W_["low_limit"].values

# avg(成交均价)复权口径自检：若与close比值随时间漂移，说明avg未复权，退化为(open+close)/2
_ratio = pd.Series(np.nanmedian(AVG / CLOSE, axis=1), index=dates)
if abs(_ratio.iloc[:120].median() / _ratio.iloc[-120:].median() - 1) > 0.03:
    print("警告：avg与close口径不一致，改用(open+close)/2作为次日均价")
    AVG = (OPEN + CLOSE) / 2

ON = np.full((T, NALL), np.nan, np.float32)        # 隔夜收益 open_t/close_{t-1}-1
DAY = (CLOSE / OPEN - 1).astype(np.float32)        # 日间收益 close_t/open_t-1
C2C = np.full((T, NALL), np.nan, np.float32)       # 全天收益
ON[1:] = OPEN[1:] / CLOSE[:-1] - 1
C2C[1:] = CLOSE[1:] / CLOSE[:-1] - 1
for _a in (ON, DAY, C2C):
    _a[~np.isfinite(_a)] = np.nan

# 上市/ST/一字板掩码
info = RAW["info"].reindex(codes)
_dn = dates.values.astype("datetime64[D]")[:, None]
_st = pd.to_datetime(info["start_date"]).values.astype("datetime64[D]")[None, :]
_ed = pd.to_datetime(info["end_date"]).values.astype("datetime64[D]")[None, :]
AGE_OK = (_dn >= _st + np.timedelta64(CFG["MIN_LIST_DAYS"], "D")) & (_dn <= _ed)
ST = RAW["st"].reindex(index=dates, columns=codes).fillna(False).values
LOCKED = (((OPEN >= HL * 0.999) & (CLOSE >= HL * 0.999)) | ((OPEN <= LL * 1.001) & (CLOSE <= LL * 1.001)))
BASE_OK = AGE_OK & ~ST & np.isfinite(CLOSE) & ~PAUSED           # t日可入选股票池
TRADE_OK = ~PAUSED & ~LOCKED & np.isfinite(AVG) & np.isfinite(OPEN)   # 该日可买入(非停牌、非一字板)
print("=" * 70)
print(f"【数据】股票 {NALL} 只，交易日 {T} 天（{dates[0].date()} ~ {dates[-1].date()}），耗时 {time.time()-t0:.0f}s")
print(f"一字涨跌停占比 {LOCKED.mean():.3%}（若接近0，说明聚宽前复权下涨跌停价口径不一致，请检查）")
print("=" * 70)

# 滚动窗口有效性：窗口内收益全部有限，且停牌占比不超限
_fin = np.isfinite(ON) & np.isfinite(DAY) & np.isfinite(C2C)
CS_FIN = np.cumsum(_fin, axis=0, dtype=np.int32)
CS_PAUSE = np.cumsum(PAUSED, axis=0, dtype=np.int32)
del _fin


def window_data(t, W, cand):
    """返回 t日(含)往前W日窗口内完全有效的股票下标及其隔夜/日间/全天收益窗口。"""
    if t < W:
        return None
    ok = cand & ((CS_FIN[t] - CS_FIN[t - W]) == W) & ((CS_PAUSE[t] - CS_PAUSE[t - W]) <= CFG["HALT_MAX_WIN"] * W)
    idx = np.where(ok)[0]
    if len(idx) < 30:
        return None
    sl = slice(t - W + 1, t + 1)
    Ow, Dw, Cw = ON[sl][:, idx], DAY[sl][:, idx], C2C[sl][:, idx]
    keep = (Ow.std(0) > 1e-7) & (Dw.std(0) > 1e-7) & (Cw.std(0) > 1e-7)
    if keep.sum() < 30:
        return None
    return idx[keep], Ow[:, keep], Dw[:, keep], Cw[:, keep]


def zs(X):
    X = X - X.mean(0, keepdims=True)
    return X / np.maximum(X.std(0, keepdims=True), 1e-8)


def xcorr(X, Y):
    """C[i,j] = corr(X_i, Y_j)；X取隔夜、Y取日间时 C[i,j] 表示 i 的隔夜收益对 j 的日间收益的预测相关性。"""
    return (zs(X).T @ zs(Y)) / X.shape[0]


# ==========================================================================
# 二、群组识别：KMeans(研报改进) 与 d-LE-SC(文献原方法)
# ==========================================================================
def dlesc(A, m=4, eta=0.3, seed=0):
    """定向谱聚类 d-LE-SC（图4、图5）：厄米矩阵主特征向量 -> [Re,Im] -> KMeans；迭代更新SBM参数eta。"""
    At = A.T
    lead = lag = None
    for _ in range(m):
        eta = float(np.clip(eta, 0.02, 0.48))
        H = (1j * np.log((1 - eta) / eta) * (A - At) + np.log(1 / (4 * eta * (1 - eta))) * (A + At)).astype(np.complex64)
        if A.shape[0] <= 1500:
            _, V = np.linalg.eigh(H); v = V[:, -1]
        else:
            _, V = eigsh(H, k=1, which="LA"); v = V[:, 0]
        lab = KMeans(2, n_init=5, random_state=seed).fit_predict(np.column_stack([v.real, v.imag]))
        g0, g1 = np.where(lab == 0)[0], np.where(lab == 1)[0]
        if len(g0) == 0 or len(g1) == 0:
            break
        f01, f10 = A[np.ix_(g0, g1)].sum(), A[np.ix_(g1, g0)].sum()
        lead, lag = (g0, g1) if f01 >= f10 else (g1, g0)
        eta = min(f01, f10) / (f01 + f10 + 1e-9)
    if lead is None:
        raise ValueError("d-LE-SC聚类退化")
    return lead, lag


def split_groups(Aabs, method="kmeans", seed=0):
    """
    Aabs[i,j]=|corr(隔夜_i, 日间_j)|，对角线为0，作为有向加权邻接矩阵。
    kmeans：特征=[行(作为领先方的强度分布) ‖ 列(作为滞后方的强度分布)]，PCA降维后KMeans(k=2)，
            再用平均流强度 S_{a->b}=mean(A[a,b]) 判定 S更大的方向 a->b 为 领先->滞后（图6的argmax）。
    """
    if method == "dlesc":
        try:
            return dlesc(Aabs, seed=seed)
        except Exception:
            pass
    N = Aabs.shape[0]
    feats = np.hstack([Aabs, Aabs.T])
    Z = PCA(n_components=min(CFG["PCA_DIM"], N - 1), svd_solver="randomized", random_state=seed).fit_transform(feats)
    lab = KMeans(2, n_init=5, random_state=seed).fit_predict(Z)
    g0, g1 = np.where(lab == 0)[0], np.where(lab == 1)[0]
    if len(g0) < 2 or len(g1) < 2:
        return g0, g1
    v0 = (lab == 0).astype(np.float32); v1 = 1 - v0
    s01 = float(v0 @ Aabs @ v1) / (len(g0) * len(g1))
    s10 = float(v1 @ Aabs @ v0) / (len(g0) * len(g1))
    return (g0, g1) if s01 >= s10 else (g1, g0)


def membership_mask(index_code):
    """指数成分(point-in-time，每月首个交易日取一次成分，月内沿用)。"""
    if index_code is None:
        return np.ones((T, NALL), bool)
    pos = {c: i for i, c in enumerate(codes)}
    m = np.zeros((T, NALL), bool)
    firsts = pd.Series(np.arange(T), index=dates).groupby([dates.year, dates.month]).first().values
    for k, ti in enumerate(firsts):
        try:
            mem = get_index_stocks(index_code, date=dates[ti].date())
        except Exception:
            mem = []
        cols = [pos[c] for c in mem if c in pos]
        end = firsts[k + 1] if k + 1 < len(firsts) else T
        if cols:
            m[ti:end, cols] = True
    return m


def diag_group_split(member, n_days=40):
    """复现研报"d-LE-SC在沪深300内常出现某一群组不足5只"的现象：对比两种聚类的最小群组规模。"""
    ts = [t for t in np.linspace(max(CFG["WINDOW"], dates.searchsorted(CFG["DAILY_START"])), T - 3, n_days).astype(int)]
    res = {"dlesc": [], "kmeans": []}
    for t in ts:
        wd = window_data(t, CFG["WINDOW"], member[t] & BASE_OK[t])
        if wd is None:
            continue
        A = np.abs(xcorr(wd[1], wd[2])); np.fill_diagonal(A, 0)
        for mth in res:
            lead, lag = split_groups(A, mth, seed=t)
            res[mth].append(min(len(lead), len(lag)))
    for mth, v in res.items():
        v = np.array(v)
        print(f"  [{mth}] 最小群组均值 {v.mean():.1f} 只；<5只的比例 {np.mean(v < 5):.1%}（抽样{len(v)}天）")


# ==========================================================================
# 三、日度多空组合回测（三种持仓方案）
# ==========================================================================
def _fwd_returns():
    F = {k: np.full((T, NALL), np.nan, np.float32) for k in SCHEMES}
    F["o2o"][:-2] = OPEN[2:] / OPEN[1:-1] - 1
    F["day"][:-1] = DAY[1:]
    F["c2c"][:-1] = C2C[1:]
    return F


FWD = _fwd_returns()


def run_daily_backtest(uni_name, member, cluster_every):
    W = CFG["WINDOW"]
    start_t = max(W + 1, dates.searchsorted(pd.Timestamp(CFG["DAILY_START"])))
    lead_flag = np.full(NALL, -1, np.int8)
    recs = []
    for t in range(start_t, T - 2):
        wd = window_data(t, W, member[t] & BASE_OK[t])
        if wd is None or len(wd[0]) < CFG["MIN_N_DAILY"]:
            continue
        idx, Ow, Dw, _ = wd
        A = xcorr(Ow, Dw); np.fill_diagonal(A, 0)
        Aabs = np.abs(A)
        if (t - start_t) % cluster_every == 0 or not (lead_flag >= 0).any():
            lead_i, lag_i = split_groups(Aabs, CFG["CLUSTER_METHOD"], seed=t)
            lead_flag[:] = -1
            lead_flag[idx[lead_i]] = 1; lead_flag[idx[lag_i]] = 0
        else:
            lf = lead_flag[idx]
            lead_i, lag_i = np.where(lf == 1)[0], np.where(lf == 0)[0]
        if len(lead_i) < 5 or len(lag_i) < 5:
            continue
        # 图6：LeadScore_i=sum_j A_ij；取前50%领先股票的平均隔夜收益为方向信号
        lead_score = Aabs[lead_i].sum(1)
        t_lead = lead_i[lead_score >= np.median(lead_score)]
        sig = float(np.nanmean(ON[t, idx[t_lead]]))
        # LagScore_j = sum_{i in lead} M_ij(带符号)：受领先群组正/负影响的程度
        lag_score = A[np.ix_(lead_i, lag_i)].sum(0)
        gid = idx[lag_i]
        can_trade = TRADE_OK[t + 1][gid]
        rec = dict(date=dates[t + 1], sig=sig, n_lead=len(lead_i), n_lag=len(lag_i))
        for s in SCHEMES:
            r = FWD[s][t, gid]
            m = can_trade & np.isfinite(r)
            if m.sum() < 10:
                rec[f"{s}_top"] = rec[f"{s}_bot"] = np.nan
                continue
            order = np.argsort(lag_score[m]); rr = r[m]
            k = max(1, int(0.2 * m.sum()))
            rec[f"{s}_top"], rec[f"{s}_bot"] = rr[order[-k:]].mean(), rr[order[:k]].mean()
        recs.append(rec)
        if len(recs) % 250 == 0:
            print(f"  [{uni_name}] 日度回测进度 {dates[t].date()}  已完成 {len(recs)} 天")
    return pd.DataFrame(recs).set_index("date") if recs else None


def daily_nav_frame(rec, s):
    x = rec[["sig", f"{s}_top", f"{s}_bot"]].dropna()
    top, bot = x[f"{s}_top"].values, x[f"{s}_bot"].values
    d = np.where(x["sig"].values > 0, 1, -1)
    long_leg, short_leg = np.where(d > 0, top, bot), np.where(d > 0, bot, top)
    ls = long_leg - short_leg
    return pd.DataFrame({"多头组": (1 + long_leg).cumprod(), "空头组": (1 + short_leg).cumprod(),
                         "多空组": (1 + ls).cumprod(), "空多组": (1 - ls).cumprod()}, index=x.index), ls


def plot_daily(nav, uni, s):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    nav[["多头组", "空头组"]].plot(ax=ax[0], color=["#1f6fa8", "#e6c24a"])
    ax[0].set_title(f"{uni}_滞后组_多空净值表现一（{SCHEMES[s]}）", fontsize=10)
    nav[["多空组", "空多组"]].plot(ax=ax[1], color=["#1f6fa8", "#e6c24a"])
    ax[1].set_title(f"{uni}_滞后组_多空净值表现二（{SCHEMES[s]}）", fontsize=10)
    for a in ax:
        a.grid(alpha=0.3); a.set_xlabel("")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_daily_{uni}_{s}.png", dpi=120); plt.close(fig)


def plot_group_counts(rec, uni):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.stackplot(rec.index, rec["n_lead"], rec["n_lag"], labels=["群组1(领先)", "群组2(滞后)"], colors=["#1f6fa8", "#e6c24a"])
    ax.set_title(f"{uni}成分股群组分类" if uni != "全市场" else "全市场股票群组分类"); ax.legend(loc="upper left")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_groups_{uni}.png", dpi=120); plt.close(fig)


def run_daily_section():
    print("\n【三、实证研究】群组识别与日度多空组合回测")
    rows = []
    for uni, ic in CFG["DAILY_UNIVERSES"]:
        print(f"\n--- 股票池：{uni} ---")
        member = membership_mask(ic)
        if uni == "沪深300":
            print("d-LE-SC 与 KMeans 的群组规模对比（研报：d-LE-SC在沪深300内会出现群组不足5只）")
            diag_group_split(member)
        rec = run_daily_backtest(uni, member, CFG["CLUSTER_EVERY"].get(uni, 1))
        if rec is None or len(rec) < 30:
            print(f"{uni} 有效样本不足，跳过"); continue
        rec.to_csv(f"{OUT_DIR}/daily_records_{uni}.csv", encoding="utf-8-sig")
        plot_group_counts(rec, uni)
        for s in SCHEMES:
            nav, ls = daily_nav_frame(rec, s)
            n = len(ls)
            ann_ls = nav["多空组"].iloc[-1] ** (252 / n) - 1
            ann_sl = nav["空多组"].iloc[-1] ** (252 / n) - 1
            rows.append(dict(股票池=uni, 持仓方案=SCHEMES[s], 样本天数=n, 多空年化=ann_ls, 空多年化=ann_sl,
                             多头组终值=nav["多头组"].iloc[-1], 空头组终值=nav["空头组"].iloc[-1],
                             空多日胜率=float(np.mean(ls < 0))))
            plot_daily(nav, uni, s)
    if rows:
        tab = pd.DataFrame(rows)
        print("\n日度回测汇总（研报：全市场次日开盘调仓 空多年化约8.81%；中证1000约10.51%；沪深300区分度不突出；\n"
              "A股呈反转效应=空头组强于多头组，仅日间收益时区分度更明显）")
        print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
        tab.to_csv(f"{OUT_DIR}/daily_summary.csv", index=False, encoding="utf-8-sig")


# ==========================================================================
# 四、因子构建
# ==========================================================================
def topk_mean(M, k, largest=True):
    """按列取最大/最小k个(排除对角线)的均值，并返回行下标k×N。"""
    X = M.copy()
    np.fill_diagonal(X, -np.inf if largest else np.inf)
    idx = np.argpartition(X, -k, axis=0)[-k:] if largest else np.argpartition(X, k - 1, axis=0)[:k]
    return np.take_along_axis(M, idx, axis=0).mean(0), idx


def _sgn(x):
    return 1.0 if x > 0 else -1.0


def compute_factors(t):
    """在调仓信号日t(窗口含t)计算全部因子，返回 {因子名: 长度NALL的向量(池外为NaN)}。"""
    W, K = CFG["FACTOR_WINDOW"], CFG["TOPK"]
    wd = window_data(t, W, BASE_OK[t])
    if wd is None:
        return None
    idx, Ow, Dw, Cw = wd
    n = len(idx); k = min(K, n - 1)
    A = xcorr(Ow, Dw)                 # 隔夜_i -> 日间_j
    Cr = xcorr(Cw, Cw)                # 常规(全天)相关性
    out = {}

    def put(name, vec, pos=None):
        a = np.full(NALL, np.nan, np.float32)
        a[idx if pos is None else idx[pos]] = vec
        out[name] = a

    # ---- 表5/6：全市场，不区分领先滞后 ----
    put("corr_lag_60", (A.sum(0) - np.diag(A)) / (n - 1))
    o21, d21 = zs(Ow[-21:]), zs(Dw[-21:])
    put("corr_lag_21", ((o21.sum(1) @ d21) / 21 - (o21 * d21).mean(0)) / (n - 1))
    t50, it = topk_mean(A, k, True)
    b50, ib = topk_mean(A, k, False)
    put("factor_top50", t50); put("factor_bottom50", b50); put("factor_net", t50 - b50)
    Aa = np.abs(A)
    ta, _ = topk_mean(Aa, k, True); ba, _ = topk_mean(Aa, k, False)
    put("factor_top50_abs", ta); put("factor_bottom50_abs", ba); put("factor_net_abs", ta - ba)
    # ---- 表7/8：常规相关性、以及用隔夜日间相关性筛选股票后的常规相关性 ----
    t50c, _ = topk_mean(Cr, k, True); b50c, _ = topk_mean(Cr, k, False)
    put("factor_top50_allday", t50c); put("factor_bottom50_allday", b50c); put("factor_net_allday", t50c - b50c)
    put("factor_top50_corr_by_lag", np.take_along_axis(Cr, it, axis=0).mean(0))
    put("factor_small50_corr_by_lag", np.take_along_axis(Cr, ib, axis=0).mean(0))
    # ---- 表1-4：领先/滞后群组因子 ----
    A0 = np.abs(A); np.fill_diagonal(A0, 0)
    lead_i, lag_i = split_groups(A0, CFG["CLUSTER_METHOD"], seed=t)
    if len(lead_i) >= K and len(lag_i) >= K:
        # 表1/2：信号源=领先群组(前50%)，待选池=滞后群组；方向=信号源当日隔夜收益均值的符号
        ls_ = A0[lead_i].sum(1); tl = lead_i[ls_ >= np.median(ls_)]
        put("factor_lead", A[np.ix_(lead_i, lag_i)].mean(0) * _sgn(np.nanmean(ON[t, idx[tl]])), lag_i)
        lg_ = A0[lag_i].sum(1); tg = lag_i[lg_ >= np.median(lg_)]
        put("factor_lag", A[np.ix_(lag_i, lead_i)].mean(0) * _sgn(np.nanmean(ON[t, idx[tg]])), lead_i)
        # 表3/4：领先群组中相关性靠前50只作信号源，方向由过去5/21日涨跌幅确定
        sub = A0[np.ix_(lead_i, lag_i)].sum(1)
        top = lead_i[np.argsort(-sub)[:K]]
        base = A[np.ix_(top, lag_i)].mean(0)
        sub2 = np.abs(Cr[np.ix_(lead_i, lag_i)]).sum(1)          # 全天(常规)相关性
        top2 = lead_i[np.argsort(-sub2)[:K]]
        base2 = Cr[np.ix_(top2, lag_i)].mean(0)
        for kk in (5, 21):
            sg1 = _sgn(np.nansum(np.nanmean(ON[t - kk + 1:t + 1][:, idx[top]], axis=1)))
            sg2 = _sgn(np.nansum(np.nanmean(C2C[t - kk + 1:t + 1][:, idx[top2]], axis=1)))
            put(f"factor_top50_lead_signal_{kk}", base * sg1, lag_i)
            put(f"factor_top50_allday_lead_signal_{kk}", base2 * sg2, lag_i)
    return out


读取本地缓存 ./outputs/market_cache_2024-01-01_2026-10-20_0.pkl
【数据】股票 5327 只，交易日 675 天（2024-01-02 ~ 2026-10-20），耗时 0s
一字涨跌停占比 0.622%（若接近0，说明聚宽前复权下涨跌停价口径不一致，请检查）


In [4]:
def rebal_indices(freq):
    s = pd.Series(np.arange(T), index=dates)
    if freq == "M":
        g = s.groupby([dates.year, dates.month]).last()
    else:
        wk = dates.to_period("W-SUN")   # ← 改：用周期分组代替isocalendar，旧版pandas也支持
        g = s.groupby(wk).last()
    first = max(CFG["FACTOR_WINDOW"], dates.searchsorted(pd.Timestamp(CFG["FACTOR_START"])))
    return [int(i) for i in g.values if i >= first and i + 1 < T]


def preprocess(x):
    """MAD去极值 + Z-Score。"""
    med = np.median(x)
    mad = np.median(np.abs(x - med)) * 1.4826
    if mad < 1e-12:
        return None
    x = np.clip(x, med - CFG["MAD_K"] * mad, med + CFG["MAD_K"] * mad)
    sd = x.std()
    return None if sd < 1e-12 else (x - x.mean()) / sd


class FactorStore:
    def __init__(self, rebal_all):
        self.rows = {t: i for i, t in enumerate(rebal_all)}
        self.panels = {n: np.full((len(rebal_all), NALL), np.nan, np.float32) for n in FACTOR_NAMES if n != "corr_combined1"}

    def fill(self, t, facs):
        r = self.rows[t]
        for n, v in facs.items():
            self.panels[n][r] = v

    def z(self, name, t, mask):
        """截面预处理后的因子值(池外NaN)；corr_combined1 = 两个低相关因子的等权(各自标准化)。"""
        out = np.full(NALL, np.nan, np.float32)
        if name == "corr_combined1":
            a = self.z("factor_top50_allday", t, mask); b = self.z("factor_small50_corr_by_lag", t, mask)
            v = 0.5 * (a + b); m = np.isfinite(v)
        else:
            v = self.panels[name][self.rows[t]]; m = mask & np.isfinite(v)
        if m.sum() < 30:
            return out
        zz = preprocess(v[m])
        if zz is not None:
            out[m] = zz
        return out


def date_ctx(t, tn):
    """调仓信号日t，次日(t+1)均价买入，下一调仓日次日(tn+1)均价卖出。"""
    b, s = t + 1, tn + 1
    r = AVG[s] / AVG[b] - 1
    mask = BASE_OK[t] & TRADE_OK[b] & np.isfinite(r)
    return mask, r


def spearman(a, b):
    return float(np.corrcoef(rankdata(a), rankdata(b))[0, 1])


def ann(x, ppy):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return np.nan if len(x) == 0 else float(np.prod(1 + x) ** (ppy / len(x)) - 1)


def evaluate(store, rebal, ppy, names, keep=()):
    per = list(zip(rebal[:-1], rebal[1:]))
    ctx = [date_ctx(t, tn) for t, tn in per]
    res, detail = {}, {}
    for name in names:
        ics, items = [], []
        for (t, tn), (mask, r) in zip(per, ctx):
            z = store.z(name, t, mask)
            m = np.isfinite(z) & np.isfinite(r)
            if m.sum() < 100:
                ics.append(np.nan); items.append(None); continue
            ics.append(spearman(z[m], r[m])); items.append((np.where(m)[0], z[m], r[m]))
        ic = np.array(ics)
        if np.isfinite(ic).sum() < 3:
            continue
        mic = np.nanmean(ic); sign = 1.0 if mic >= 0 else -1.0
        G = np.full((len(per), CFG["N_GROUP"]), np.nan); TO = np.zeros_like(G)
        prev = [None] * CFG["N_GROUP"]; mem = []
        for k, it in enumerate(items):
            if it is None:
                mem.append(None); continue
            ids, z, r = it
            parts = np.array_split(np.argsort(-sign * z, kind="mergesort"), CFG["N_GROUP"])   # ← 改：旧版numpy用mergesort代替stable；group_1=最优组
            for g, p in enumerate(parts):
                G[k, g] = r[p].mean(); cur = ids[p]
                if prev[g] is not None:
                    TO[k, g] = 1 - np.isin(prev[g], cur).mean()
                prev[g] = cur
            mem.append((ids[parts[0]], ids[parts[-1]]) if name in keep else None)
        fee = CFG["FEE"]
        long_net = G[:, 0] - fee * TO[:, 0]
        ls = long_net - (G[:, -1] + fee * TO[:, -1])        # 空头腿同样计卖出费用(保守)
        win = np.nanmean((ic * sign)[np.isfinite(ic)] > 0)
        res[name] = dict(RANK_IC=mic, ICIR=mic / (np.nanstd(ic) + 1e-12), IC胜率=win,
                         多空年化=ann(ls, ppy), 多头年化=ann(long_net, ppy), 样本期数=int(np.isfinite(ic).sum()))
        detail[name] = dict(ic=pd.Series(ic, index=[dates[t] for t, _ in per]), G=G, TO=TO, sign=sign, mem=mem, per=per,
                            net=G - fee * TO, ls=ls)
    return res, detail


def print_perf_tables(res_m, res_w):
    allrows = []
    for tname, names in TABLES.items():
        rows = []
        for n in names:
            for fq, res in (("月度", res_m), ("周度", res_w)):
                if n not in res:
                    continue
                r = res[n]
                row = {"因子名": n, "换仓频率": fq,
                       "RANK_IC%": r["RANK_IC"] * 100, "ICIR": r["ICIR"], "IC胜率%": r["IC胜率"] * 100,
                       "多空年化%": r["多空年化"] * 100, "多头年化%": r["多头年化"] * 100, "期数": r["样本期数"]}
                ref = REPORT.get(n, {}).get(fq)
                if ref:
                    row.update({"研报IC%": ref[0], "研报ICIR": ref[1], "研报胜率%": ref[2], "研报多空%": ref[3], "研报多头%": ref[4]})
                rows.append(row)
        if rows:
            df = pd.DataFrame(rows)
            print(f"\n{tname}（本次复现 vs 研报）")
            print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
            df.to_csv(f"{OUT_DIR}/{tname}.csv", index=False, encoding="utf-8-sig")
            allrows.append(df)
    if allrows:
        pd.concat(allrows).to_csv(f"{OUT_DIR}/factor_performance_all.csv", index=False, encoding="utf-8-sig")


# ---------------- 表9：因子内部相关性 ----------------
def table9(store, rebal_m):
    names = ["corr_lag_21", "corr_lag_60", "factor_top50_allday", "factor_bottom50_allday",
             "factor_net_allday", "factor_top50_corr_by_lag", "factor_small50_corr_by_lag"]
    mats = []
    for t in rebal_m[:-1]:
        mask, r = date_ctx(t, rebal_m[rebal_m.index(t) + 1])
        Z = pd.DataFrame({n: store.z(n, t, mask) for n in names})
        mats.append(Z.corr().values)
    C = pd.DataFrame(np.nanmean(np.stack(mats), axis=0), index=names, columns=names)
    print("\n表9：因子内部相关性（月度，各期截面相关系数的时间均值，%）")
    print((C * 100).round(1).to_string())
    C.to_csv(f"{OUT_DIR}/表9_因子相关性.csv", encoding="utf-8-sig")


# ---------------- 表13：与风格因子相关性 ----------------
def table13(store, rebal_m):
    if jq_get_factor_values is None:
        print("\n表13：未检测到聚宽因子库(jqfactor)，跳过风格因子相关性"); return
    names = ["corr_combined1", "factor_top50_allday", "factor_bottom50_allday", "factor_net_allday",
             "factor_top50_corr_by_lag", "factor_small50_corr_by_lag"]
    acc = {n: {k: [] for k in STYLE_MAP} for n in names}
    for i, t in enumerate(rebal_m[:-1]):
        try:
            d = jq_get_factor_values(codes, list(STYLE_MAP.values()), end_date=dates[t].date(), count=1)
            sty = {k: d[v].iloc[-1].reindex(codes).values.astype(float) for k, v in STYLE_MAP.items()}
        except Exception as e:
            print("风格因子获取失败，跳过表13：", str(e)[:100]); return
        mask, _ = date_ctx(t, rebal_m[i + 1])
        for n in names:
            z = store.z(n, t, mask)
            for k, sv in sty.items():
                m = np.isfinite(z) & np.isfinite(sv)
                if m.sum() < 100:
                    continue
                sz = preprocess(sv[m])
                if sz is not None:
                    acc[n][k].append(np.corrcoef(z[m], sz)[0, 1])
    tab = pd.DataFrame({n: {k: np.mean(v) * 100 if v else np.nan for k, v in acc[n].items()} for n in names}).T
    print("\n表13：因子与风格因子的相关性（%，聚宽风格因子库近似Barra CNE5）")
    print(tab.round(1).to_string())
    tab.to_csv(f"{OUT_DIR}/表13_风格因子相关性.csv", encoding="utf-8-sig")


# ---------------- 图24-31、表11/12：组合因子详情 ----------------
def bench_daily():
    try:
        b = get_price(CFG["BENCH"], start_date=CFG["DATA_START"], end_date=CFG["END"], frequency="daily", fields=["close"])
        b.index = pd.to_datetime(b.index)
        return b["close"].reindex(dates).pct_change()
    except Exception as e:
        print("基准指数获取失败，以全体可交易股票等权收益代替：", str(e)[:80])
        return pd.Series(np.nanmean(C2C, axis=1), index=dates)


def daily_nav_from_detail(det):
    per, TO, fee = det["per"], det["TO"], CFG["FEE"]
    f_long, f_short, fee_s = np.ones(T), np.ones(T), np.zeros(T)
    for k, (t, tn) in enumerate(per):
        if det["mem"][k] is None:
            continue
        b, s = t + 1, tn + 1
        for side, ids in enumerate(det["mem"][k]):
            arr = f_long if side == 0 else f_short
            cl = CLOSE[b:s][:, ids]
            prev = np.vstack([AVG[b, ids][None, :], cl[:-1]])
            r = np.nanmean(cl / prev - 1, axis=1)                  # b..s-1 日：当日买入均价->收盘，其后收盘到收盘
            arr[b:s] *= 1 + np.nan_to_num(r)
            arr[s] *= 1 + np.nan_to_num(np.nanmean(AVG[s, ids] / CLOSE[s - 1, ids] - 1))   # 调仓日按均价卖出
            if k + 1 < len(per):
                if side == 0:
                    arr[s] *= 1 - fee * TO[k + 1, 0]
                else:
                    fee_s[s] += fee * TO[k + 1, -1]
    lo, hi = per[0][0] + 1, per[-1][1] + 1
    sl = slice(lo, hi + 1)
    rl = pd.Series(f_long[sl] - 1, index=dates[sl])
    rs = pd.Series(f_short[sl] - 1, index=dates[sl])
    ls = rl - rs - pd.Series(fee_s[sl], index=dates[sl])
    return rl, rs, ls


def annual_table(r_long, bench):
    rows = []
    for y, r in r_long.groupby(r_long.index.year):
        nav = (1 + r).cumprod()
        ret = nav.iloc[-1] - 1
        vol = r.std() * np.sqrt(252)
        b = (1 + bench.reindex(r.index).fillna(0)).prod() - 1
        rows.append({"年份": f"{y}" + ("(不满全年)" if len(r) < 200 else ""), "年化收益率": ret, "年化波动率": vol,
                     "夏普比": ret / vol if vol > 0 else np.nan, "最大回撤": float((1 - nav / nav.cummax()).max()),
                     "指数收益": b, "超额收益": ret - b})
    return pd.DataFrame(rows).set_index("年份")


def detail_report(det, freq_name, fig_tag, bench):
    ic, G, TO = det["ic"], det["G"], det["TO"]
    net = det["net"]
    cum = np.nanprod(1 + net, axis=0) - 1
    rl, rs, ls = daily_nav_from_detail(det)
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    ax[0, 0].bar(ic.index, ic.values, width=12 if freq_name == "月度" else 4, color="#1f6fa8", label="RANK_IC")
    a2 = ax[0, 0].twinx(); a2.plot(ic.index, ic.fillna(0).cumsum().values, color="#e6c24a", label="IC累计(右轴)")
    ax[0, 0].set_title(f"图{fig_tag[0]}：因子IC值信息")
    ax[0, 1].bar(range(1, 11), cum, color="#1f6fa8")
    a3 = ax[0, 1].twinx(); a3.plot(range(1, 11), np.nanmean(TO, axis=0), color="#e6c24a")
    ax[0, 1].set_xticks(range(1, 11)); ax[0, 1].set_xticklabels([f"group_{i}" for i in range(1, 11)], rotation=45)
    ax[0, 1].set_title(f"图{fig_tag[1]}：因子分组平均收益统计(柱:累计收益, 线:平均换手率)")
    ax[1, 0].plot(rl.index, (1 + rl).cumprod(), color="#1f6fa8", label="多头累计净值")
    ax[1, 0].plot(rs.index, (1 + rs).cumprod(), color="#e6c24a", label="空头累计净值")
    ax[1, 0].legend(); ax[1, 0].set_title(f"图{fig_tag[2]}：因子多头空头累计净值")
    ax[1, 1].plot(ls.index, (1 + ls).cumprod(), color="#888888", label="多空累计净值")
    ax[1, 1].legend(); ax[1, 1].set_title(f"图{fig_tag[3]}：因子多空累计净值")
    for a in ax.ravel():
        a.grid(alpha=0.3)
    fig.suptitle(f"corr_combined1 {freq_name}换仓"); fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/fig_corr_combined1_{freq_name}.png", dpi=120); plt.close(fig)
    tab = annual_table(rl, bench)
    print(f"\n表{ '11' if freq_name == '月度' else '12'}：corr_combined1 {freq_name}换仓 分年度绩效（多头组，含卖出费用）")
    print(tab.to_string(float_format=lambda x: f"{x:.2%}" if abs(x) < 5 else f"{x:.2f}"))
    tab.to_csv(f"{OUT_DIR}/表{'11' if freq_name == '月度' else '12'}_分年度绩效.csv", encoding="utf-8-sig")


In [5]:
def run_factor_section():
    print("\n【四、因子构建研究】")
    rebal_m, rebal_w = rebal_indices("M"), rebal_indices("W")
    rebal_all = sorted(set(rebal_m) | set(rebal_w))
    cache = f"{OUT_DIR}/factor_panels_{CFG['FACTOR_START']}_{CFG['END']}_{CFG['SAMPLE_N']}_{CFG['CLUSTER_METHOD']}.pkl"
    store = FactorStore(rebal_all)
    if os.path.exists(cache):
        print("读取因子缓存", cache)
        store.panels = pd.read_pickle(cache)
    else:
        t0 = time.time()
        for i, t in enumerate(rebal_all):
            f = compute_factors(t)
            if f:
                store.fill(t, f)
            if i % 20 == 0:
                print(f"  因子计算 {i + 1}/{len(rebal_all)}  {dates[t].date()}  累计{time.time() - t0:.0f}s")
        pd.to_pickle(store.panels, cache)
    print(f"月度调仓 {len(rebal_m)} 期，周度调仓 {len(rebal_w)} 期")
    res_m, det_m = evaluate(store, rebal_m, 12, FACTOR_NAMES, keep=("corr_combined1",))
    res_w, det_w = evaluate(store, rebal_w, 52, FACTOR_NAMES, keep=("corr_combined1",))
    print_perf_tables(res_m, res_w)
    table9(store, rebal_m)
    if "corr_combined1" in det_m:
        bench = bench_daily()
        detail_report(det_m["corr_combined1"], "月度", (24, 25, 26, 27), bench)
        detail_report(det_w["corr_combined1"], "周度", (28, 29, 30, 31), bench)
    if CFG["RUN_STYLE"]:
        table13(store, rebal_m)


# ==========================================================================
BASE_OK &= membership_mask("000852.XSHG")   # ← 新增：因子/日度的股票池统一限定为中证1000成分(point-in-time)
if CFG["RUN_DAILY"]:
    run_daily_section()
if CFG["RUN_FACTOR"]:
    run_factor_section()
print("\n全部完成，结果保存在", OUT_DIR)


【三、实证研究】群组识别与日度多空组合回测

--- 股票池：中证1000 ---
  [中证1000] 日度回测进度 2025-04-17  已完成 250 天
  [中证1000] 日度回测进度 2026-04-29  已完成 500 天

日度回测汇总（研报：全市场次日开盘调仓 空多年化约8.81%；中证1000约10.51%；沪深300区分度不突出；
A股呈反转效应=空头组强于多头组，仅日间收益时区分度更明显）
多头组终值    多空年化               持仓方案  样本天数   空多年化  空多日胜率  空头组终值     股票池
1.2461 -0.1378      次日开盘买入-后日开盘卖出   612 0.1262 0.4984 1.7410  中证1000
1.6084 -0.1109  次日开盘买入-当日收盘卖出(日间)   612 0.0971 0.5114 2.0849  中证1000
1.2872 -0.0932   理想：当日收盘买入-次日收盘卖出   612 0.0721 0.4788 1.5917  中证1000

【四、因子构建研究】
读取因子缓存 ./outputs/factor_panels_2024-04-01_2026-10-20_0_kmeans.pkl
月度调仓 30 期，周度调仓 132 期

表2_隔夜相关性因子（本次复现 vs 研报）
ICIR  IC胜率%  RANK_IC%          因子名  多头年化%  多空年化% 换仓频率   期数  研报IC%  研报ICIR  研报多头%  研报多空%  研报胜率%
0.20  41.38      2.40  factor_lead  16.60   0.47   月度   29   0.29    0.03   1.40  -1.50  49.20
-0.14  52.67     -1.94  factor_lead   5.43  -5.45   周度  131   1.30    0.13   3.00   9.50  56.50
-0.18  44.83     -2.17   factor_lag  18.16  23.39   月度   29   0.13    0.01   3.50  -2.70  49.20
-0.01  